# caliper dev on Colab

The GPU CI. Run after a push that touches anything below the ports: builds the
Rust core + extension, runs the no-GPU suites, then the on-device (`l2`/`l6`)
suites, then `caliper doctor`, `caliper fingerprint --check`, and `caliper selftest --full`. Paste the pass/fail tail into the PR.

Runtime: pick a GPU runtime (A100 preferred; T4 also works). The bootstrap cell
installs Rust + nsys; a released wheel needs neither, but a from-source dev run
does.

In [ ]:
# bootstrap: Rust toolchain + nsys, then the package from source
!curl -sSf https://sh.rustup.rs | sh -s -- -y >/dev/null && source $HOME/.cargo/env
!apt-get -qq install -y nsight-systems-cli 2>/dev/null || true
%cd /content
![ -d caliper ] || git clone --depth 1 https://github.com/mansoor-mamnoon/caliper
%cd caliper
!git pull --ff-only
!pip -q install -e ".[dev]"
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
# no-GPU suites (must already be green from ci-cpu, re-run as a smoke check)
!source $HOME/.cargo/env && cargo test --all --all-features 2>&1 | tail -20
!pytest -m "l0 or l1" -q 2>&1 | tail -20

In [ ]:
# on-device suites: L2 oracles, L4 unlocked reproducibility, L6 end-to-end
!pytest -m "l2 or l4 or l6" -q 2>&1 | tail -40 || true
!source $HOME/.cargo/env && CALIPER_GPU_PORTS=real cargo test -p caliper-gpu --features cuda 2>&1 | tail -20 || true


In [ ]:
# acceptance checks: doctor + fingerprint completeness + the oracle self-test
!caliper doctor || true
!caliper fingerprint --check || true
!caliper selftest --full --json | tee /content/selftest-report.json | python -m json.tool | tail -40


In [ ]:
# regression compare smoke: the committed fixtures exercise playbook #12
!caliper compare --baseline tests/testdata/base.parquet --candidate tests/testdata/slow.parquet --fail-on-regression || echo "exit $?"
